# **Bronze Layer**

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable as T
from pyspark.sql.window import Window

In [0]:
%run /Workspace/fmcg_pipeline/setup_fmcg_data/utilities


In [0]:
dbutils.widgets.text("catalog", "lakehouse", "Catalog")
dbutils.widgets.text("datasource", "gross_price", "Datasource")
catlog =dbutils.widgets.get("catalog")
dataSource = dbutils.widgets.get("datasource")

## data source is a folder name ofs3 customer data , if folder name change we just change it in a widget


In [0]:
print(catlog)
print(dataSource)

lakehouse
gross_price


In [0]:
path = f's3://fmcg-child-data/{dataSource}/*.csv'
df_bronze = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path).select("*","_metadata.file_name","_metadata.file_size")
df_bronze = df_bronze.withColumn("timestamp", F.current_timestamp())

#df_bronze.printSchema()
#df_bronze.show()


In [0]:
df_bronze.write.format("delta")\
.mode("append")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{bronze_schema}.{dataSource}")

#to enable change data feed and you can track the changes or audit the changes (enableChangeDataFeed)

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:139)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:139)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
%sql
select * from lakehouse.bronze.gross_price

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
%sql
select distinct gross_price from lakehouse.bronze.gross_price

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

# **silver layer**

In [0]:
df_silver = df_bronze.withColumn(
    "gross_price",
    F.when(F.col("gross_price").isin("unknown", "not_available"), None)
     .otherwise(F.col("gross_price").cast("double"))
)

df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price") < 0, -F.col("gross_price"))
     .otherwise(F.col("gross_price"))
)


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
%sql 
select * from lakehouse.silver.gross_price

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
avg_gross_price = df_silver.select(F.avg("gross_price")).first()[0]
avg_gross_price = round(avg_gross_price, 2)
print(avg_gross_price)
df_silver = df_silver.withColumn("gross_price", F.round(F.col("gross_price"), 2))
df_silver = df_silver.withColumn(
    "gross_price",
    F.when(F.col("gross_price").isNull(), avg_gross_price)
     .otherwise(F.col("gross_price"))
)

display(df_silver)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
#product_mapping = [88888888, 99999999, 77777777]

#valid_products = (
   # df.filter(~F.col("product_id").isin(*product_mapping))
   #   .groupBy("gross_price")
     # .agg(F.first("product_id").alias("real_product_id"))
#)

#df = (
   # df.join(valid_products, on="gross_price", how="left")
      #.withColumn(
       #   "product_id",
        #  F.when(
         #     F.col("product_id").isin(*product_mapping),
         #     F.col("real_product_id")
         # ).otherwise(F.col("product_id"))
     # )
      #.drop("real_product_id")
#)

#df.show()


In [0]:
df_silver = df_silver.drop_duplicates()

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
display(df_silver)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy"]

df_silver = df_silver.withColumn(
    "month",
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
display(df_silver)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
%sql 
drop table if exists lakehouse.silver.gross_price

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
df_silver.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{silver_schema}.{dataSource}")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
%sql
select * from lakehouse.silver.gross_price

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
df_products = spark.read.table("lakehouse.silver.products")
df_products = df_products.select("product_id", "product" , "product_code")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
display(df_products)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
df1 = df_products.join(df_silver, on="product_id", how="left")
df1 = df1.filter(F.col("product").isNotNull())
df1 = df1.select("product_id", "product", "product_code", "month", "gross_price" , "file_size" , "timestamp" ,"file_name")
df1 = df1.dropDuplicates()
display(df1)

product_id,product,product_code,month,gross_price,file_size,timestamp,file_name
25891601,SportsBar Electrolyte Mix Lemon-Lime,1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025-11-01,440.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891101,SportsBar Energy Bar Choco Fudge,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025-09-01,84.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891201,SportsBar Protien Bar Peanut Crunch,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,2025-10-01,108.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891401,SportsBar Greek Yogurt Pro Vanilla,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025-07-01,74.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891502,SportsBar Oats Cookie Bites ChocoChip,cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025-08-01,203.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891603,SportsBar Electrolyte Mix Lemon-Lime,1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025-08-01,171.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891302,SportsBar Granola Crunch Honey Almond,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,2025-08-01,207.79,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891502,SportsBar Oats Cookie Bites ChocoChip,cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025-10-01,237.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891101,SportsBar Energy Bar Choco Fudge,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025-11-01,83.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891403,SportsBar Greek Yogurt Pro Vanilla,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025-12-01,50.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv


In [0]:
%sql
drop table if exists lakehouse.silver.gross_price

In [0]:
df1.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{silver_schema}.{dataSource}")

# **Glod Layer**

In [0]:
df_silver = spark.read.table(f"{catlog}.{silver_schema}.{dataSource}")
display(df_silver)
df_silver = df_silver.dropDuplicates()


product_id,product,product_code,month,gross_price,file_size,timestamp,file_name
25891601,SportsBar Electrolyte Mix Lemon-Lime,1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025-11-01,440.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891101,SportsBar Energy Bar Choco Fudge,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025-09-01,84.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891201,SportsBar Protien Bar Peanut Crunch,1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,2025-10-01,108.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891401,SportsBar Greek Yogurt Pro Vanilla,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025-07-01,74.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891502,SportsBar Oats Cookie Bites ChocoChip,cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025-08-01,203.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891603,SportsBar Electrolyte Mix Lemon-Lime,1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025-08-01,171.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891302,SportsBar Granola Crunch Honey Almond,10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,2025-08-01,207.79,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891502,SportsBar Oats Cookie Bites ChocoChip,cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025-10-01,237.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891101,SportsBar Energy Bar Choco Fudge,8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025-11-01,83.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv
25891403,SportsBar Greek Yogurt Pro Vanilla,8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025-12-01,50.0,2741,2026-03-01T19:16:21.329Z,gross_price.csv


In [0]:
df1 = df1.select(
    "product_code",
    F.year(df1.month).alias("year"),
    df1["gross_price"].cast("int").alias("price_inr")
)

display(df1)


product_code,year,price_inr
1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025,440
8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025,84
1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,2025,108
8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025,74
cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025,203
1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025,171
10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,2025,207
cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025,237
8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025,83
8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025,50


In [0]:
%sql
drop table if exists lakehouse.gold.sp_gross_price

In [0]:
df1.write.format("delta")\
.mode("overwrite")\
.option("delta.enableChangeDataFeed", "true")\
.saveAsTable(f"{catlog}.{gold_schema}.{f'sp_{dataSource}'}")

# **Merged tables**

In [0]:
delta_table = T.forName(spark,"lakehouse.gold.dim_gross_price")
display(delta_table)


In [0]:
child_table = spark.read.table(f"{catlog}.{gold_schema}.sp_gross_price")
display(child_table)

product_code,year,price_inr
1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025,440
8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025,84
1ddc933eee438043ce9e80beb1158330ad3e178d243d91e0c63c2c11e74ef1ef,2025,108
8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025,74
cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025,203
1f663d5bfbd69949522ad3dfcc52f1da0c9d17eb1ed2580fac2f190ad76c77da,2025,171
10258f662d8ddd2905c0330260e0bb1f1d27c8924bfc762ead9e9def6c264217,2025,207
cc4694228861e0c9a2f3f5df8c8423ff2504ebb89b90427263b181ecc13b6aff,2025,237
8571bb108439429a0a346dff9ae19cb278ed781fc16f49161a6eb5ffa6339373,2025,83
8a363a7229c72ac69962b84049656c352127ee4cbc5a6b1827144d55e8995ac7,2025,50


In [0]:
child_table=child_table.dropDuplicates(["product_code"])

In [0]:
delta_table.alias("target").merge(
    source=child_table.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#delta_table=delta_table.toDF().dropDuplicates()